In [ ]:
# @title 1. Setup - Mount Drive and Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content/drive/MyDrive/BT5151_toxic_comment_agent')
sys.path.insert(0, '/content/drive/MyDrive/BT5151_toxic_comment_agent')

import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "langgraph", "gradio", "openai",
    "transformers", "datasets", "evaluate", "accelerate",
    "sentence-transformers", "scikit-learn", "seaborn",
])
print("Setup complete.")


In [ ]:
# @title 2. Imports
from pipeline.state import (
    BuildState, RuntimeState,
    build_initial_build_state, build_initial_runtime_state,
)
from pipeline.build import compile_build_graph
from pipeline.graph import compile_runtime_graph
from pipeline.controller import run
print("Imports OK.")


In [ ]:
# @title 3. API Keys via Colab Secrets
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("OPENAI_API_KEY set from Colab Secrets.")


In [ ]:
# @title 4. Build Layer - Train Models (run once)
PROJECT_ROOT = "/content/drive/MyDrive/BT5151_toxic_comment_agent"
build_state = build_initial_build_state(PROJECT_ROOT)
print("Starting build layer...")
build_result = run(mode="build", state=build_state)
print(f"Build complete. Selected model: {build_result.get('selected_model_id')}")
print(f"Justification: {build_result.get('selection_justification')}")


In [ ]:
# @title 5. Build Layer - Inspect Outputs
import json
from pathlib import Path

print("=== Preprocessing Summary ===")
print(json.dumps(build_result.get("preprocessing_summary", {}), indent=2))

print("\n=== Candidate Models Trained ===")
print(build_result.get("candidate_model_ids"))

print("\n=== Selected Model ===")
print(build_result.get("selected_model_id"))

from IPython.display import Image, display
figures_dir = Path(PROJECT_ROOT) / "experiments" / "figures"
for fig in sorted(figures_dir.glob("*.png")):
    print(f"\n{fig.name}")
    display(Image(str(fig)))


In [ ]:
# @title 6. Runtime Layer - Sample Comments
SAMPLE_COMMENTS = [
    "Thank you for your edits, this article is much clearer now.",
    "This is a stupid comment and your argument makes no sense.",
    "You are an absolute idiot and nobody wants you here.",
    "What the hell is going on with this page?",
]

runtime_results = []
for comment in SAMPLE_COMMENTS:
    state = build_initial_runtime_state(comment, project_root=PROJECT_ROOT)
    result = run(mode="moderate", state=state)
    runtime_results.append(result)
    print(f"\nComment: {comment[:60]}...")
    print(f"  Predicted: {result.get('predicted_label')} (prob={result.get('toxicity_probability'):.3f})")
    print(f"  Severity:  {result.get('severity_label')}")
    print(f"  Action:    {result.get('action_label')}")
    print(f"  Warning skipped: {result.get('warning_skipped')}")


In [ ]:
# @title 7. Runtime Layer - Warning Messages
for comment, result in zip(SAMPLE_COMMENTS, runtime_results):
    warning = result.get("warning_message", "")
    if warning:
        print(f"\nComment: {comment[:60]}...")
        print(f"Action:  {result.get('action_label')}")
        print(f"Warning:\n  {warning}")
        print("-" * 60)


In [ ]:
# @title 8. LangGraph - Graph Definitions
build_g = compile_build_graph()
print("Build graph nodes:", list(build_g.graph.nodes.keys()))

runtime_g = compile_runtime_graph()
print("Runtime graph nodes:", list(runtime_g.graph.nodes.keys()))

from pipeline.controller import run as controller_run
print("Controller: run(mode='build'|'moderate', state)")


In [ ]:
# @title 9. SKILL.md Contents
from pathlib import Path

for skill_path in sorted(Path("SKILLS").glob("**/*.md")):
    print(f"\n{'='*60}")
    print(f"SKILL: {skill_path.parent.name}")
    print('='*60)
    print(skill_path.read_text())


In [ ]:
# @title 10. Gradio Demo
from app import build_demo
demo = build_demo()
demo.launch(share=True)
